[구글 코랩(Colab)에서 실행하기](https://colab.research.google.com/github/lovedlim/bigdata_analyst_cert_v2/blob/main/part2/ch7/ch7_ex_multi_class_classification.ipynb)

In [20]:
# 1. 문제정의
# 평가: f1 macro
# target: Credit_Score
# 최종파일: result.csv(컬럼 1개 pred)

# 2. 라이브러리 및 데이터 불러오기
import pandas as pd
import sklearn
import lightgbm
import xgboost

train = pd.read_csv("score_train.csv")
test = pd.read_csv("score_test.csv")

y = train.pop('Credit_Score')

print(train.shape, test.shape)
n_train = len(train)
combined = pd.concat([train, test])
cols = combined.select_dtypes('object').columns
cols

# from sklearn.preprocessing import LabelEncoder

# le_y = LabelEncoder()
# y = le_y.fit_transform(y)

# for col in cols:
#     le = LabelEncoder()
#     combined[col] = le.fit_transform(combined[col].astype(str))

combined = pd.get_dummies(combined)

train = combined[:n_train]
test = combined[n_train:]
train.shape, test.shape

# 정규화
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
train = scaler.fit_transform(train)
test = scaler.transform(test)

# 분할
from sklearn.model_selection import train_test_split
# sklearn.model_selection.__all__
X_tri, X_val, y_tri, y_val = train_test_split(train, y, test_size=0.2, random_state=0)

# 훈련 및 평가
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBRFClassifier

lg = RandomForestClassifier(random_state=0)
lg.fit(X_tri, y_tri)
pred = lg.predict(X_val)

from sklearn.metrics import f1_score
f1 = f1_score(y_val, pred, average='macro')
print(f1)

# lgb: 0.6796364247249338
# rf: 0.6820137656352824 / 원핫인코딩 0.7020460066061172
# xgb: 0.6939783664397962

# 예측 및 저장
pred = lg.predict(test)
result = pd.DataFrame({'pred':pred})
result.to_csv('result.csv',index=False)

(4198, 20) (1499, 20)
0.7020460066061172


# Section1

### 베이스라인

In [37]:
# 1. 문제정의
# 평가: f1 macro
# target: Credit_Score
# 최종파일: result.csv(컬럼 1개 pred)

# 2. 라이브러리 및 데이터 불러오기
import pandas as pd

# train = pd.read_csv("score_train.csv")
# test = pd.read_csv("score_test.csv")
train = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part2/ch7/score_train.csv")
test = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part2/ch7/score_test.csv")

# 3. 탐색적 데이터 분석(EDA)
print("===== 데이터 크기 =====")
print("Train Shape:", train.shape)
print("Test Shape:", test.shape)
print("\n") # 줄 바꿈

print("===== 데이터 정보(자료형) =====")
print(train.info())
print("\n")

print("===== train 결측치 수 =====")
print(train.isnull().sum().sum())
print("\n")

print("===== test 결측치 수 =====")
print(test.isnull().sum().sum())
print("\n")

print("===== target 빈도 =====")
print(train['Credit_Score'].value_counts())

===== 데이터 크기 =====
Train Shape: (4198, 21)
Test Shape: (1499, 20)


===== 데이터 정보(자료형) =====
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4198 entries, 0 to 4197
Data columns (total 21 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Delay_from_due_date       4198 non-null   float64
 1   Num_of_Delayed_Payment    4198 non-null   float64
 2   Num_Credit_Inquiries      4198 non-null   float64
 3   Credit_Utilization_Ratio  4198 non-null   float64
 4   Credit_History_Age        4198 non-null   float64
 5   Payment_of_Min_Amount     4198 non-null   object 
 6   Amount_invested_monthly   4198 non-null   float64
 7   Monthly_Balance           4198 non-null   float64
 8   Credit_Mix                4198 non-null   object 
 9   Payment_Behaviour         4198 non-null   object 
 10  Age                       4198 non-null   float64
 11  Annual_Income             4198 non-null   float64
 12  Num_Bank_Accounts         

In [38]:
# 4. 데이터 전처리
# 원핫인코딩 (target컬럼이 object형이라 제외)
target = train.pop('Credit_Score')

train = pd.get_dummies(train)
test = pd.get_dummies(test)

# 5. 검증 데이터 분할
from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split(train, target, test_size=0.2, random_state=0)

print("\n ===== 분할된 데이터 크기 =====")
print(X_tr.shape, X_val.shape, y_tr.shape, y_val.shape)

# 6. 머신러닝 학습 및 평가
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(random_state=0)
rf.fit(X_tr, y_tr)
pred = rf.predict(X_val)

from sklearn.metrics import f1_score
f1 = f1_score(y_val, pred, average='macro')
print('\n f1-macro:', f1)

# 7. 예측 및 결과 파일 생성
pred = rf.predict(test)
submit = pd.DataFrame({'pred':pred})
submit.to_csv("result.csv", index=False)

# 제출파일 확인
print("\n ===== 제출파일 (샘플 5개) =====")
print(pd.read_csv("result.csv").head())


 ===== 분할된 데이터 크기 =====
(3358, 29) (840, 29) (3358,) (840,)

 f1-macro: 0.7004593488873695

 ===== 제출파일 (샘플 5개) =====
       pred
0      Poor
1      Good
2  Standard
3      Good
4  Standard


### 성능개선

In [39]:
# 2. 라이브러리 및 데이터 불러오기
import pandas as pd

# train = pd.read_csv("score_train.csv")
# test = pd.read_csv("score_test.csv")
train = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part2/ch7/score_train.csv")
test = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part2/ch7/score_test.csv")

# 4. 데이터 전처리
target = train.pop('Credit_Score')

# 스케일링
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
cols = train.select_dtypes(include=['int', 'float']).columns
train[cols] = scaler.fit_transform(train[cols])
test[cols] = scaler.transform(test[cols])

# 원핫인코딩
train = pd.get_dummies(train)
test = pd.get_dummies(test)

# 레이블 인코딩
# target = train.pop('Credit_Score')
# from sklearn.preprocessing import LabelEncoder
# cols = train.select_dtypes(include='object').columns
# for col in cols:
#     le = LabelEncoder()
#     train[col] = le.fit_transform(train[col])
#     test[col] = le.transform(test[col])

# 5. 검증 데이터 분할
from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split(train, target, test_size=0.2, random_state=0)

# 6. 머신러닝 학습 및 평가
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(random_state=0)
rf.fit(X_tr, y_tr)
pred = rf.predict(X_val)

from sklearn.metrics import f1_score
f1 = f1_score(y_val, pred, average='macro')
print('f1-macro:', f1)

# 7. 예측 및 결과 파일 생성
pred = rf.predict(test)
submit = pd.DataFrame({'pred':pred})
submit.to_csv("result.csv", index=False)

f1-macro: 0.7020460066061172


# Section2

### 베이스라인

In [40]:
# 1. 문제정의
# 평가: f1-macro
# target: Drug
# 최종파일: result.csv(컬럼 1개 pred, 1확률값)

# 2. 라이브러리 및 데이터 불러오기
import pandas as pd
# train = pd.read_csv("drug_train.csv")
# test = pd.read_csv("drug_test.csv")
train = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part2/ch7/drug_train.csv")
test = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part2/ch7/drug_test.csv")

# 3. 탐색적 데이터 분석(EDA)
print("===== 데이터 정보(자료형) =====")
print(train.info())

print("\n ===== train 결측치 수 =====")
print(train.isnull().sum().sum())

print("\n ===== test 결측치 수 =====")
print(test.isnull().sum().sum())

print("\n ===== train/test 카테고리별 수 =====")
print(train[['Sex', 'BP', 'Cholesterol']].nunique())
print(test[['Sex', 'BP', 'Cholesterol']].nunique())

print("\n ===== target 빈도 =====")
print(train['Drug'].value_counts())

===== 데이터 정보(자료형) =====
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Age          100 non-null    int64  
 1   Sex          100 non-null    object 
 2   BP           100 non-null    object 
 3   Cholesterol  100 non-null    object 
 4   Na_to_K      100 non-null    float64
 5   Drug         100 non-null    object 
dtypes: float64(1), int64(1), object(4)
memory usage: 4.8+ KB
None

 ===== train 결측치 수 =====
0

 ===== test 결측치 수 =====
0

 ===== train/test 카테고리별 수 =====
Sex            2
BP             3
Cholesterol    2
dtype: int64
Sex            2
BP             3
Cholesterol    2
dtype: int64

 ===== target 빈도 =====
Drug
DrugY    41
drugX    34
drugA    13
drugB     8
drugC     4
Name: count, dtype: int64


In [41]:
# 4. 데이터 전처리
# 원핫인코딩
target = train.pop('Drug')
train = pd.get_dummies(train)
test = pd.get_dummies(test)

# 5. 검증 데이터 분할
from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split(train, target, test_size=0.2, random_state=0)

# 6. 머신러닝 학습 및 평가
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(random_state=0)
rf.fit(X_tr, y_tr)
pred = rf.predict(X_val)

from sklearn.metrics import f1_score
f1 = f1_score(y_val, pred, average='macro')
print('\n f1-macro:', f1)

# 7. 예측 및 결과 파일 생성
pred = rf.predict(test)
submit = pd.DataFrame({'pred':pred})
submit.to_csv("result.csv", index=False)


 f1-macro: 1.0


In [42]:
# 크로스 밸리데이션(cross-validation)
from sklearn.metrics import f1_score
from sklearn.model_selection import cross_val_score
f1_scores = cross_val_score(rf, train, target, cv=3, scoring='f1_macro')
print(f1_scores)
print(f1_scores.mean())

[1.         0.93777778 0.78461538]
0.9074643874643874


In [43]:
# scoring에 입력 가능한 평가지표 확인 방법
from sklearn.metrics import get_scorer_names
print(get_scorer_names())

['accuracy', 'adjusted_mutual_info_score', 'adjusted_rand_score', 'average_precision', 'balanced_accuracy', 'completeness_score', 'd2_absolute_error_score', 'explained_variance', 'f1', 'f1_macro', 'f1_micro', 'f1_samples', 'f1_weighted', 'fowlkes_mallows_score', 'homogeneity_score', 'jaccard', 'jaccard_macro', 'jaccard_micro', 'jaccard_samples', 'jaccard_weighted', 'matthews_corrcoef', 'mutual_info_score', 'neg_brier_score', 'neg_log_loss', 'neg_max_error', 'neg_mean_absolute_error', 'neg_mean_absolute_percentage_error', 'neg_mean_gamma_deviance', 'neg_mean_poisson_deviance', 'neg_mean_squared_error', 'neg_mean_squared_log_error', 'neg_median_absolute_error', 'neg_negative_likelihood_ratio', 'neg_root_mean_squared_error', 'neg_root_mean_squared_log_error', 'normalized_mutual_info_score', 'positive_likelihood_ratio', 'precision', 'precision_macro', 'precision_micro', 'precision_samples', 'precision_weighted', 'r2', 'rand_score', 'recall', 'recall_macro', 'recall_micro', 'recall_samples'

### 성능개선

In [44]:
# 2. 라이브러리 및 데이터 불러오기
import pandas as pd
# train = pd.read_csv("drug_train.csv")
# test = pd.read_csv("drug_test.csv")
train = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part2/ch7/drug_train.csv")
test = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part2/ch7/drug_test.csv")

# 4. 데이터 전처리
target = train.pop('Drug')

# 스케일링
# from sklearn.preprocessing import MinMaxScaler
# scaler = MinMaxScaler()
# train['Age'] = scaler.fit_transform(train[['Age']])
# test['Age'] = scaler.transform(test[['Age']])

# 원핫인코딩 (Drug 컬럼 제외)
train = pd.get_dummies(train)
test = pd.get_dummies(test)

# 레이블 인코딩 (Drug 컬럼 제외)
# target = train.pop('Drug')
# from sklearn.preprocessing import LabelEncoder
# cols = train.select_dtypes(include='object').columns
# for col in cols:
#     le = LabelEncoder()
#     train[col] = le.fit_transform(train[col])
#     test[col] = le.transform(test[col])

# 5. 크로스 밸리데이션(cross-validation)
from sklearn.metrics import f1_score
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=0)
f1_scores = cross_val_score(rf, train, target, cv=3, scoring='f1_macro')
print(f1_scores.mean())

# 6. 머신러닝 학습 및 평가
rf.fit(train, target)

# 7. 예측 및 결과 파일 생성
pred = rf.predict(test)
submit = pd.DataFrame({'pred':pred})
submit.to_csv("result.csv", index=False)

0.9074643874643874


# Section3

### 베이스라인

In [45]:
# 1. 문제정의
# 평가: f1-weighted
# target: Type
# 최종파일: result.csv(컬럼 1개 pred)

# 2. 라이브러리 및 데이터 불러오기
import pandas as pd
# train = pd.read_csv("glass_train.csv")
# test = pd.read_csv("glass_test.csv")
train = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part2/ch7/glass_train.csv")
test = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part2/ch7/glass_test.csv")

# 3. 탐색적 데이터 분석(EDA)
print("===== 데이터 크기 =====")
print(train.shape, test.shape)

print("\n ===== train 데이터 샘플 =====")
print(train.head(1))

print("\n ===== test 데이터 샘플 =====")
print(test.head(1))

print("\n ===== 데이터 정보(자료형) =====")
print(train.info())

print("\n ===== train 결측치 수 =====")
print(train.isnull().sum().sum())

print("\n ===== test 결측치 수 =====")
print(test.isnull().sum().sum())

print("\n ===== target 빈도 =====")
print(train['Type'].value_counts())

===== 데이터 크기 =====
(149, 10) (65, 9)

 ===== train 데이터 샘플 =====
        RI     Na    Mg    Al     Si    K    Ca   Ba   Fe  Type
0  1.51829  14.46  2.24  1.62  72.38  0.0  9.26  0.0  0.0     6

 ===== test 데이터 샘플 =====
        RI     Na    Mg    Al     Si     K    Ca   Ba    Fe
0  1.51748  12.86  3.56  1.27  73.21  0.54  8.38  0.0  0.17

 ===== 데이터 정보(자료형) =====
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 149 entries, 0 to 148
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   RI      149 non-null    float64
 1   Na      149 non-null    float64
 2   Mg      149 non-null    float64
 3   Al      149 non-null    float64
 4   Si      149 non-null    float64
 5   K       149 non-null    float64
 6   Ca      149 non-null    float64
 7   Ba      149 non-null    float64
 8   Fe      149 non-null    float64
 9   Type    149 non-null    int64  
dtypes: float64(9), int64(1)
memory usage: 11.8 KB
None

 ===== train 결측치 수 =====
0

 =

In [46]:
# 4. 데이터 전처리
target = train.pop('Type')

# 5. 검증 데이터 분할
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(train, target, test_size=0.2, random_state=0)

# 6. 머신러닝 학습 및 평가
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(random_state=0)
rf.fit(X_tr, y_tr)
pred = rf.predict(X_val)

from sklearn.metrics import f1_score
score = f1_score(y_val, pred, average='weighted')
print('f1:', score)

# 7. 예측 및 결과 파일 생성
pred = rf.predict(test)
submit = pd.DataFrame({'pred':pred})
submit.to_csv("result.csv", index=False)

f1: 0.6119801766860591


### 성능개선

In [47]:
# 2. 라이브러리 및 데이터 불러오기
import pandas as pd
# train = pd.read_csv("glass_train.csv")
# test = pd.read_csv("glass_test.csv")
train = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part2/ch7/glass_train.csv")
test = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part2/ch7/glass_test.csv")

# 4. 데이터 전처리
target = train.pop('Type')

# 스케일링 효과 없음

# 5. 검증 데이터 분할
from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split(train, target, test_size=0.2, random_state=0)

# 6. 머신러닝 학습 및 평가
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(max_depth=5, n_estimators=200, random_state=0)
rf.fit(X_tr, y_tr)
pred = rf.predict(X_val)

from sklearn.metrics import f1_score
score = f1_score(y_val, pred, average='weighted')
print('f1:', score)

# 7. 예측 및 결과 파일 생성
pred = rf.predict(test)
submit = pd.DataFrame({'pred':pred})
submit.to_csv("result.csv", index=False)

f1: 0.6507936507936507
